# Cálculo de SPEI

### 1. Importar data

In [ ]:
import pandas as pd
import numpy as np

# Cargar los datos limpios 
try:
    df_mensual = pd.read_csv('data/processed/datos_mensuales_limpios.csv')
    df_metadata = pd.read_csv('data/processed/metadata_estaciones_finales.csv')
    
    # Convertir la columna de fecha de nuevo a datetime
    df_mensual['fecha'] = pd.to_datetime(df_mensual['fecha'])
    
    print("Datos y metadatos cargados exitosamente.")
    print("\nInformación de los datos mensuales:")
    print(df_mensual.info())
    print("\nInformación de los metadatos:")
    print(df_metadata.info())
    
except FileNotFoundError:
    print("Error")

Datos y metadatos cargados exitosamente.

Información de los datos mensuales:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3168 entries, 0 to 3167
Data columns (total 6 columns):
 #   Column        Non-Null Count  Dtype         
---  ------        --------------  -----         
 0   estacion_id   3168 non-null   object        
 1   fecha         3168 non-null   datetime64[ns]
 2   lluvia_total  3154 non-null   float64       
 3   temp_media    3154 non-null   float64       
 4   temp_max      3154 non-null   float64       
 5   temp_min      3154 non-null   float64       
dtypes: datetime64[ns](1), float64(4), object(1)
memory usage: 148.6+ KB
None

Información de los metadatos:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 11 entries, 0 to 10
Data columns (total 5 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   estacion_id  11 non-null     object 
 1   latitud      11 non-null     float64
 2   longitud     11 non-null     flo

### 2. Cálculo del SPEI

In [2]:
from scipy.stats import loglaplace # La distribución Log-Logística en SciPy se llama loglaplace

# Unir los datos para tener la latitud 
df_completo = pd.merge(df_mensual, df_metadata, on='estacion_id')

# Función para calcular la ETP de Thornthwaite 
def thornthwaite_manual(df_input):
    t_avg = df_input['temp_media']
    lat = df_input['latitud']
    month = df_input['fecha'].dt.month
    
    t_avg_clipped = t_avg.clip(0)
    i = (t_avg_clipped / 5) ** 1.514
    
    df_temp = pd.DataFrame({'estacion_id': df_input['estacion_id'], 'i': i})
    I = df_temp.groupby('estacion_id')['i'].transform('sum')

    a = (6.75e-7 * I**3) - (7.71e-5 * I**2) + (1.792e-2 * I) + 0.49239

    dias_mes = pd.Series(df_input['fecha'].dt.days_in_month, index=t_avg.index)
    
    k_table = {1:0.92, 2:0.82, 3:1.03, 4:1.12, 5:1.20, 6:1.21, 7:1.22, 8:1.15, 9:1.05, 10:0.99, 11:0.92, 12:0.90}
    factor_k = month.map(k_table)

    etp_sin_ajustar = 16 * ((10 * t_avg_clipped) / I) ** a
    etp_final = etp_sin_ajustar * factor_k * (dias_mes / 30)
    
    return etp_final

# Aplicar la función y calcular el balance hídrico 
df_completo['pet'] = thornthwaite_manual(df_completo)
df_completo['balance_hidrico'] = df_completo['lluvia_total'] - df_completo['pet']

# Calcular el SPEI
df_agrupado = df_completo.set_index('fecha').groupby('estacion_id')
lista_spei_final = []

print("--- Calculando SPEI manualmente con SciPy ---")
for estacion_nombre, datos_estacion in df_agrupado:
    
    # Rellenar NaNs en el balance hídrico antes de empezar
    balance = datos_estacion['balance_hidrico'].fillna(0)
    
    for escala in [3, 6, 12]:
        # Acumular el balance hídrico en la escala de tiempo
        balance_acumulado = balance.rolling(window=escala, min_periods=escala).sum()
        
        # Omitir los NaNs iniciales para el ajuste
        series_a_ajustar = balance_acumulado.dropna()
        
        # Ajustar la distribución Log-Logística a los datos
        params = loglaplace.fit(series_a_ajustar)
        
        # Calcular la CDF (Cumulative Distribution Function) para toda la serie acumulada
        cdf = loglaplace.cdf(balance_acumulado, *params)
        
        # Transformar la CDF a valores estandarizados (Z-scores)
        # Esto es el SPEI. ppf es la Percent Point Function (inversa de la CDF) de una Normal
        from scipy.stats import norm
        spei_values = norm.ppf(cdf)
        
        datos_estacion[f'spei_{escala}'] = spei_values
        
    lista_spei_final.append(datos_estacion)
    print(f"SPEI calculado para la estación: {estacion_nombre}")

df_final_con_spei = pd.concat(lista_spei_final).reset_index()

print("\n--- Cálculo del SPEI finalizado ---")
print(df_final_con_spei[['estacion_id', 'fecha', 'lluvia_total', 'temp_media', 'spei_3', 'spei_6', 'spei_12']].tail())

--- Calculando SPEI manualmente con SciPy ---
SPEI calculado para la estación: CAAZ
SPEI calculado para la estación: CMEZA
SPEI calculado para la estación: CONC
SPEI calculado para la estación: COVIE
SPEI calculado para la estación: GBRU
SPEI calculado para la estación: LUQUE
SPEI calculado para la estación: MCAL
SPEI calculado para la estación: MINGA
SPEI calculado para la estación: PILAR
SPEI calculado para la estación: PJC
SPEI calculado para la estación: SEST

--- Cálculo del SPEI finalizado ---
     estacion_id      fecha  lluvia_total  temp_media    spei_3    spei_6  \
3163        SEST 2023-08-31          96.9   22.619355 -1.169400 -0.424226   
3164        SEST 2023-09-30         141.3   24.970000 -0.576959 -0.957047   
3165        SEST 2023-10-31         171.5   25.880645  0.149855 -0.961624   
3166        SEST 2023-11-30         404.6   26.603333  1.197518  0.657434   
3167        SEST 2023-12-31         197.0   27.835484  1.300908  1.086742   

       spei_12  
3163  0.937784 

### Código de Diagnóstico para Recalcular y Comparar el SPEI

In [ ]:
from scipy.stats import loglaplace, norm

# Asumimos que 'df_mensual' y 'df_metadata' están cargados 
# df_mensual = pd.read_csv('datos_mensuales_limpios.csv')
# df_metadata = pd.read_csv('metadata_estaciones_finales.csv')
# df_mensual['fecha'] = pd.to_datetime(df_mensual['fecha'])

# REUTILIZAMOS LA FUNCIÓN THORNTHWAITE
def thornthwaite_manual(df_input):
    t_avg = df_input['temp_media']
    lat = df_input['latitud']
    month = df_input['fecha'].dt.month
    
    t_avg_clipped = t_avg.clip(0)
    i = (t_avg_clipped / 5) ** 1.514
    
    # Necesitamos recrear el df_completo temporalmente para el transform
    df_completo_temp = pd.merge(df_mensual, df_metadata, on='estacion_id')
    df_temp = pd.DataFrame({'estacion_id': df_completo_temp['estacion_id'], 'i': i.reindex(df_completo_temp.index)})
    I = df_temp.groupby('estacion_id')['i'].transform('sum')
    I = I.loc[df_input.index] # Aseguramos que el índice coincida

    a = (6.75e-7 * I**3) - (7.71e-5 * I**2) + (1.792e-2 * I) + 0.49239

    dias_mes = pd.Series(df_input['fecha'].dt.days_in_month, index=t_avg.index)
    
    k_table = {1:0.92, 2:0.82, 3:1.03, 4:1.12, 5:1.20, 6:1.21, 7:1.22, 8:1.15, 9:1.05, 10:0.99, 11:0.92, 12:0.90}
    factor_k = month.map(k_table)

    etp_sin_ajustar = 16 * ((10 * t_avg_clipped) / I) ** a
    etp_final = etp_sin_ajustar * factor_k * (dias_mes / 30)
    
    return etp_final

# PROCESO DE DIAGNÓSTICO
print("--- INICIANDO DIAGNÓSTICO DE CÁLCULO DEL SPEI ---")

estaciones_a_comparar = ['CAAZ', 'LUQUE']
escala_a_comparar = 12 # Nos centramos en el SPEI-12 que generó el mapa

for estacion in estaciones_a_comparar:
    print(f"\n" + "="*50)
    print(f"DIAGNÓSTICO PARA LA ESTACIÓN: {estacion}")
    print("="*50)
    
    # 1. Preparar datos de la estación
    df_estacion = pd.merge(
        df_mensual[df_mensual['estacion_id'] == estacion],
        df_metadata[df_metadata['estacion_id'] == estacion],
        on='estacion_id'
    ).set_index('fecha').sort_index()

    precip_series = df_estacion['lluvia_total'].fillna(0)
    temp_series = df_estacion['temp_media'].fillna(df_estacion['temp_media'].mean())

    # 2. Calcular ETP y Balance Hídrico
    df_estacion['pet'] = thornthwaite_manual(df_estacion.reset_index()).values
    df_estacion['balance_hidrico'] = df_estacion['lluvia_total'] - df_estacion['pet']
    balance = df_estacion['balance_hidrico'].fillna(0)

    # 3. Calcular serie acumulada
    balance_acumulado = balance.rolling(window=escala_a_comparar, min_periods=escala_a_comparar).sum()
    series_a_ajustar = balance_acumulado.dropna()
    
    # 4. Ajustar la distribución
    params = loglaplace.fit(series_a_ajustar)
    
    # 5. Calcular SPEI
    cdf = loglaplace.cdf(balance_acumulado, *params)
    spei_values = norm.ppf(cdf)
    
    # IMPRIMIR VALORES INTERMEDIOS CLAVE 
    print(f"Latitud: {df_estacion['latitud'].iloc[0]}")
    print(f"Precipitación media mensual: {precip_series.mean():.2f} mm")
    print(f"Temperatura media mensual: {temp_series.mean():.2f} °C")
    print(f"ETP media mensual calculada: {df_estacion['pet'].mean():.2f} mm")
    print(f"Balance hídrico medio: {balance.mean():.2f} mm")
    print(f"\nParámetros ajustados de la distribución Log-Logística (c, loc, scale):")
    print(f"  c={params[0]:.4f}, loc={params[1]:.4f}, scale={params[2]:.4f}")
    
    print(f"\nEstadísticas de la serie acumulada (base para el ajuste):")
    print(series_a_ajustar.describe())
    
    print(f"\nSPEI-{escala_a_comparar} promedio final: {np.nanmean(spei_values):.4f}")

--- INICIANDO DIAGNÓSTICO DE CÁLCULO DEL SPEI ---

DIAGNÓSTICO PARA LA ESTACIÓN: CAAZ
Latitud: -26.17
Precipitación media mensual: 131.70 mm
Temperatura media mensual: 22.07 °C
ETP media mensual calculada: 0.00 mm
Balance hídrico medio: 131.70 mm

Parámetros ajustados de la distribución Log-Logística (c, loc, scale):
  c=5.5459, loc=-1.4412, scale=1570.0412

Estadísticas de la serie acumulada (base para el ajuste):
count     277.000000
mean     1551.712996
std       326.346924
min       820.800000
25%      1290.700000
50%      1568.600000
75%      1799.300000
max      2265.400000
Name: balance_hidrico, dtype: float64

SPEI-12 promedio final: -0.1216

DIAGNÓSTICO PARA LA ESTACIÓN: LUQUE
Latitud: -25.24
Precipitación media mensual: 118.47 mm
Temperatura media mensual: 23.20 °C
ETP media mensual calculada: 0.00 mm
Balance hídrico medio: 118.47 mm

Parámetros ajustados de la distribución Log-Logística (c, loc, scale):
  c=7.0490, loc=-1.4812, scale=1380.7812

Estadísticas de la serie acumu

El Problema:
La función thornthwaite_manual está devolviendo cero (0) para la ETP en TODOS los meses para TODAS las estaciones.

Consecuencias:

Cálculo Incorrecto del Balance Hídrico: Como la ETP es siempre cero, tu balance_hidrico es simplemente P - 0, es decir, es idéntico a la precipitación (lluvia_total).
Cálculo Incorrecto del SPEI: No estás calculando el SPEI. Estás calculando el SPI (Índice de Precipitación Estandarizada), porque has eliminado el componente de la temperatura del cálculo.

La Paradoja Resuelta:

CAAZ tiene un SPEI-12 promedio negativo (-0.1216).
LUQUE tiene un SPEI-12 promedio positivo (0.0666).
¿Por qué? Porque CAAZ tiene una precipitación promedio más alta (131.70 mm) pero también una variabilidad mucho mayor (su std en la serie acumulada es 326 vs. 259 de LUQUE). El SPI/SPEI es un índice estandarizado. CAAZ tiene más eventos de lluvia por debajo de su propia (alta) media, lo que la hace parecer "más seca" en términos relativos.
Pero todo esto es incorrecto, porque se basa en un cálculo fundamentalmente erróneo del balance hídrico.

### Solución: La Función thornthwaite_manual Corregida

In [ ]:
# FUNCIÓN THORNTHWAITE CORREGIDA 
def thornthwaite_calculo(t_avg, lat):
    """
    Calcula la ETP mensual para una ÚNICA serie de tiempo.
    t_avg: Serie de pandas de temperaturas medias mensuales (°C).
    lat: Un único valor de latitud (en grados decimales).
    """
    # Asegurarse de que el índice sea datetime
    t_avg.index = pd.to_datetime(t_avg.index)
    
    # Índice de calor mensual (i)
    t_avg_clipped = t_avg.clip(0) 
    i = (t_avg_clipped / 5) ** 1.514
    
    # Índice de calor anual (I)
    # Sumamos los 12 valores mensuales de 'i' de cada año.
    # Usamos un promedio de I para todos los años para mayor estabilidad.
    I = i.groupby(i.index.year).sum().mean()
    
    if I <= 0: return 0 # Evitar división por cero si hace mucho frío

    # Exponente 'a'
    a = (6.75e-7 * I**3) - (7.71e-5 * I**2) + (1.792e-2 * I) + 0.49239

    # Factor de corrección de la luz del día (simplificado para el hemisferio sur)
    k_table_sur = {1:0.92, 2:0.82, 3:1.03, 4:1.12, 5:1.20, 6:1.21, 7:1.22, 8:1.15, 9:1.05, 10:0.99, 11:0.92, 12:0.90}
    factor_k = t_avg.index.month.map(k_table_sur)
    
    # Días en el mes
    dias_mes = t_avg.index.days_in_month

    # ETP sin ajustar (mm/mes)
    etp_sin_ajustar = 16 * ((10 * t_avg_clipped) / I) ** a
    
    # ETP final ajustada
    etp_final = etp_sin_ajustar * factor_k * (dias_mes / 30)
    
    return etp_final

### CÁLCULO de SPEI principal con esta nueva lógica

In [ ]:

# BUCLE PRINCIPAL PARA EL CÁLCULO DEL SPEI 

print("--- Iniciando el recálculo del SPEI para cada estación ---")

# Unir los datos para un manejo más fácil
df_completo = pd.merge(df_mensual, df_metadata, on='estacion_id')
df_agrupado = df_completo.set_index('fecha').groupby('estacion_id')

lista_spei_final = []

for estacion_nombre, datos_estacion in df_agrupado:
    
    # Preparar las series de datos para la estación actual
    latitud_estacion = datos_estacion['latitud'].iloc[0]
    temp_series = datos_estacion['temp_media'].fillna(datos_estacion['temp_media'].mean())
    precip_series = datos_estacion['lluvia_total'].fillna(0)
    
    # Calcular la ETP usando la nueva función
    pet_series = thornthwaite_calculo(t_avg=temp_series, lat=latitud_estacion)
    
    # Añadir la ETP y el Balance Hídrico al DataFrame de la estación
    datos_estacion['pet'] = pet_series
    datos_estacion['balance_hidrico'] = precip_series - pet_series
    balance = datos_estacion['balance_hidrico'].fillna(0)
    
    # Calcular el SPEI para diferentes escalas
    for escala in [3, 6, 12]:
        balance_acumulado = balance.rolling(window=escala, min_periods=escala).sum()
        series_a_ajustar = balance_acumulado.dropna()
        
        # Ajustar la distribución
        params = loglaplace.fit(series_a_ajustar)
        
        # Calcular CDF y transformar a Z-scores (SPEI)
        cdf = loglaplace.cdf(balance_acumulado, *params)
        spei_values = norm.ppf(cdf)
        
        datos_estacion[f'spei_{escala}'] = spei_values
        
    lista_spei_final.append(datos_estacion)
    print(f"SPEI recalculado para la estación: {estacion_nombre}")

# Unir todos los resultados
df_final_con_spei = pd.concat(lista_spei_final).reset_index()

print("\n--- Recálculo del SPEI finalizado ---")

# Verificar las últimas filas del nuevo DataFrame
print("\nÚltimas filas del DataFrame con el SPEI recalculado:")
print(df_final_con_spei[['estacion_id', 'fecha', 'lluvia_total', 'temp_media', 'pet', 'spei_3', 'spei_6', 'spei_12']].tail())


--- Iniciando el recálculo del SPEI para cada estación ---
SPEI recalculado para la estación: CAAZ


/home/hjvargaso/proyectos/tesis_sequia/.venv/lib/python3.12/site-packages/scipy/stats/_continuous_distns.py:6735: RuntimeWarning: divide by zero encountered in power
  return cd2*x**(c-1)
/home/hjvargaso/proyectos/tesis_sequia/.venv/lib/python3.12/site-packages/scipy/stats/_continuous_distns.py:6735: RuntimeWarning: divide by zero encountered in power
  return cd2*x**(c-1)
/home/hjvargaso/proyectos/tesis_sequia/.venv/lib/python3.12/site-packages/scipy/stats/_continuous_distns.py:6735: RuntimeWarning: divide by zero encountered in power
  return cd2*x**(c-1)
/home/hjvargaso/proyectos/tesis_sequia/.venv/lib/python3.12/site-packages/scipy/stats/_continuous_distns.py:6735: RuntimeWarning: divide by zero encountered in power
  return cd2*x**(c-1)
/home/hjvargaso/proyectos/tesis_sequia/.venv/lib/python3.12/site-packages/scipy/stats/_continuous_distns.py:6735: RuntimeWarning: divide by zero encountered in power
  return cd2*x**(c-1)


SPEI recalculado para la estación: CMEZA
SPEI recalculado para la estación: CONC


/home/hjvargaso/proyectos/tesis_sequia/.venv/lib/python3.12/site-packages/scipy/stats/_continuous_distns.py:6735: RuntimeWarning: divide by zero encountered in power
  return cd2*x**(c-1)
/home/hjvargaso/proyectos/tesis_sequia/.venv/lib/python3.12/site-packages/scipy/stats/_continuous_distns.py:6735: RuntimeWarning: divide by zero encountered in power
  return cd2*x**(c-1)
/home/hjvargaso/proyectos/tesis_sequia/.venv/lib/python3.12/site-packages/scipy/stats/_continuous_distns.py:6735: RuntimeWarning: divide by zero encountered in power
  return cd2*x**(c-1)
/home/hjvargaso/proyectos/tesis_sequia/.venv/lib/python3.12/site-packages/scipy/stats/_continuous_distns.py:6735: RuntimeWarning: divide by zero encountered in power
  return cd2*x**(c-1)
/home/hjvargaso/proyectos/tesis_sequia/.venv/lib/python3.12/site-packages/scipy/stats/_continuous_distns.py:6735: RuntimeWarning: divide by zero encountered in power
  return cd2*x**(c-1)
/home/hjvargaso/proyectos/tesis_sequia/.venv/lib/python3.12/

SPEI recalculado para la estación: COVIE
SPEI recalculado para la estación: GBRU


/home/hjvargaso/proyectos/tesis_sequia/.venv/lib/python3.12/site-packages/scipy/stats/_continuous_distns.py:6735: RuntimeWarning: divide by zero encountered in power
  return cd2*x**(c-1)
/home/hjvargaso/proyectos/tesis_sequia/.venv/lib/python3.12/site-packages/scipy/stats/_continuous_distns.py:6735: RuntimeWarning: divide by zero encountered in power
  return cd2*x**(c-1)
/home/hjvargaso/proyectos/tesis_sequia/.venv/lib/python3.12/site-packages/scipy/stats/_continuous_distns.py:6735: RuntimeWarning: divide by zero encountered in power
  return cd2*x**(c-1)
/home/hjvargaso/proyectos/tesis_sequia/.venv/lib/python3.12/site-packages/scipy/stats/_continuous_distns.py:6735: RuntimeWarning: divide by zero encountered in power
  return cd2*x**(c-1)
/home/hjvargaso/proyectos/tesis_sequia/.venv/lib/python3.12/site-packages/scipy/stats/_continuous_distns.py:6735: RuntimeWarning: divide by zero encountered in power
  return cd2*x**(c-1)
/home/hjvargaso/proyectos/tesis_sequia/.venv/lib/python3.12/

SPEI recalculado para la estación: LUQUE
SPEI recalculado para la estación: MCAL


/home/hjvargaso/proyectos/tesis_sequia/.venv/lib/python3.12/site-packages/scipy/stats/_continuous_distns.py:6735: RuntimeWarning: divide by zero encountered in power
  return cd2*x**(c-1)
/home/hjvargaso/proyectos/tesis_sequia/.venv/lib/python3.12/site-packages/scipy/stats/_continuous_distns.py:6735: RuntimeWarning: divide by zero encountered in power
  return cd2*x**(c-1)
/home/hjvargaso/proyectos/tesis_sequia/.venv/lib/python3.12/site-packages/scipy/stats/_continuous_distns.py:6735: RuntimeWarning: divide by zero encountered in power
  return cd2*x**(c-1)
/home/hjvargaso/proyectos/tesis_sequia/.venv/lib/python3.12/site-packages/scipy/stats/_continuous_distns.py:6735: RuntimeWarning: divide by zero encountered in power
  return cd2*x**(c-1)


SPEI recalculado para la estación: MINGA
SPEI recalculado para la estación: PILAR
SPEI recalculado para la estación: PJC
SPEI recalculado para la estación: SEST

--- Recálculo del SPEI finalizado ---

Últimas filas del DataFrame con el SPEI recalculado:
     estacion_id      fecha  lluvia_total  temp_media         pet    spei_3  \
3163        SEST 2023-08-31          96.9   22.619355  102.317000  0.853454   
3164        SEST 2023-09-30         141.3   24.970000  120.157514  0.873574   
3165        SEST 2023-10-31         171.5   25.880645  129.777737  0.898627   
3166        SEST 2023-11-30         404.6   26.603333  126.336614  0.974262   
3167        SEST 2023-12-31         197.0   27.835484  145.479731  0.980115   

        spei_6   spei_12  
3163  0.903305  0.942949  
3164  0.877685  0.921626  
3165  0.867438  0.882908  
3166  0.939344  0.920226  
3167  0.952603  0.946888  


/home/hjvargaso/proyectos/tesis_sequia/.venv/lib/python3.12/site-packages/scipy/stats/_continuous_distns.py:6735: RuntimeWarning: divide by zero encountered in power
  return cd2*x**(c-1)
/home/hjvargaso/proyectos/tesis_sequia/.venv/lib/python3.12/site-packages/scipy/stats/_continuous_distns.py:6735: RuntimeWarning: divide by zero encountered in power
  return cd2*x**(c-1)
/home/hjvargaso/proyectos/tesis_sequia/.venv/lib/python3.12/site-packages/scipy/stats/_continuous_distns.py:6735: RuntimeWarning: divide by zero encountered in power
  return cd2*x**(c-1)
/home/hjvargaso/proyectos/tesis_sequia/.venv/lib/python3.12/site-packages/scipy/stats/_continuous_distns.py:6735: RuntimeWarning: divide by zero encountered in power
  return cd2*x**(c-1)
/home/hjvargaso/proyectos/tesis_sequia/.venv/lib/python3.12/site-packages/scipy/stats/_continuous_distns.py:6735: RuntimeWarning: divide by zero encountered in power
  return cd2*x**(c-1)


Diagnóstico del RuntimeWarning: divide by zero encountered in power
1. Significado:
Esta advertencia proviene de scipy, específicamente de la función que calcula la densidad de probabilidad (pdf) o la función de distribución acumulada (cdf) de la loglaplace (Log-Logística).
La fórmula matemática de esta distribución implica elevar un valor a una potencia. Si ese valor es cero o negativo, la operación matemática puede ser inválida (ej., log(0) no está definido, (-2)^0.5 es un número complejo), lo que resulta en una división por cero en los cálculos internos.
2. ¿Por qué?
El SPEI se basa en ajustar la serie de balance hídrico acumulado (balance_acumulado) a la distribución Log-Logística.
Esta distribución, en su formulación estándar, espera valores estrictamente positivos.
Sin embargo, nuestra serie de balance hídrico (P - ETP) y, por lo tanto, la serie acumulada, puede contener (y de hecho contendrá) muchos valores negativos.
Cuando scipy.stats.loglaplace.fit() o cdf() se encuentra con estos valores negativos o cero, lanza una advertencia porque está intentando realizar una operación matemática para la que no está diseñada.
3. ¿Afecta?
Potencialmente, sí, y de manera grave. Si la función de ajuste no puede manejar los valores negativos, los parámetros que estima (params) podrían ser incorrectos o inestables.
Si te fijas en los resultados de tu tabla final, los valores del SPEI son todos positivos y están agrupados en un rango muy estrecho (ej., 0.85 a 0.98). Esto es completamente anómalo. Un SPEI real debe tener media cero y fluctuar entre valores negativos y positivos (típicamente entre -3 y +3).
Conclusión: Las advertencias son un síntoma de un problema mayor. El ajuste de la distribución está fallando debido a los valores negativos, y los resultados del SPEI que estás obteniendo no son correctos.
La Solución: Estandarización de Pearson III
Hemos llegado a un punto de gran detalle técnico. La metodología original del SPEI de Vicente-Serrano et al. (2010) es un poco más compleja que un simple fit. Para manejar los ceros y los valores negativos, a menudo se usa un enfoque de estandarización no paramétrico o una distribución diferente que pueda manejar estos valores.
La solución más fuerte y estándar en la literatura climatológica para este problema (usada en muchas implementaciones del SPI/SPEI) es usar la distribución de Pearson Tipo III.
Vamos a ajustar el código para usar esta distribución, que es más apropiada para datos que pueden ser negativos.

In [ ]:
from scipy.stats import pearson3, norm # Importamos las distribuciones correctas

# BUCLE PRINCIPAL PARA EL CÁLCULO DEL SPEI (CON PEARSON III)

print("--- Iniciando el cálculo del SPEI con la distribución Pearson III ---")

df_completo = pd.merge(df_mensual, df_metadata, on='estacion_id')
df_agrupado = df_completo.set_index('fecha').groupby('estacion_id')

lista_spei_final = []

for estacion_nombre, datos_estacion in df_agrupado:
    
    latitud_estacion = datos_estacion['latitud'].iloc[0]
    temp_series = datos_estacion['temp_media'].fillna(datos_estacion['temp_media'].mean())
    precip_series = datos_estacion['lluvia_total'].fillna(0)
    
    pet_series = thornthwaite_calculo(t_avg=temp_series, lat=latitud_estacion)
    
    datos_estacion['pet'] = pet_series
    datos_estacion['balance_hidrico'] = precip_series - pet_series
    balance = datos_estacion['balance_hidrico'].fillna(0)
    
    # Bucle para cada escala temporal
    for escala in [3, 6, 12]:
        # Acumular el balance hídrico
        balance_acumulado = balance.rolling(window=escala, min_periods=escala).sum()
        series_a_ajustar = balance_acumulado.dropna()
        
        # AJUSTE CON LA DISTRIBUCIÓN CORRECTA 
        # 1. Ajustar la distribución de Pearson Tipo III
        # Esta distribución se define por 3 parámetros: skew (asimetría), loc (media), scale (desv. est.)
        params = pearson3.fit(series_a_ajustar)
        
        # 2. Calcular la CDF (Cumulative Distribution Function)
        cdf = pearson3.cdf(balance_acumulado, *params)
        
        # 3. Transformar la CDF a Z-scores (SPEI)
        spei_values = norm.ppf(cdf)
        
        datos_estacion[f'spei_{escala}'] = spei_values
        
    lista_spei_final.append(datos_estacion)
    print(f"SPEI calculado para la estación: {estacion_nombre}")

# Unir todos los resultados
df_final_con_spei = pd.concat(lista_spei_final).reset_index()

print("\n--- Cálculo del SPEI finalizado ---")

# Verificar las últimas filas del nuevo DataFrame
print("\nÚltimas filas del DataFrame con el SPEI correctamente calculado:")
print(df_final_con_spei[['estacion_id', 'fecha', 'lluvia_total', 'temp_media', 'pet', 'spei_3', 'spei_6', 'spei_12']].tail())


--- Iniciando el cálculo del SPEI con la distribución Pearson III ---
SPEI calculado para la estación: CAAZ
SPEI calculado para la estación: CMEZA
SPEI calculado para la estación: CONC
SPEI calculado para la estación: COVIE
SPEI calculado para la estación: GBRU
SPEI calculado para la estación: LUQUE
SPEI calculado para la estación: MCAL
SPEI calculado para la estación: MINGA
SPEI calculado para la estación: PILAR
SPEI calculado para la estación: PJC
SPEI calculado para la estación: SEST

--- Cálculo del SPEI finalizado ---

Últimas filas del DataFrame con el SPEI correctamente calculado:
     estacion_id      fecha  lluvia_total  temp_media         pet    spei_3  \
3163        SEST 2023-08-31          96.9   22.619355  102.317000 -0.706142   
3164        SEST 2023-09-30         141.3   24.970000  120.157514 -0.454075   
3165        SEST 2023-10-31         171.5   25.880645  129.777737 -0.103260   
3166        SEST 2023-11-30         404.6   26.603333  126.336614  1.245491   
3167      

### Verificacion de resultados

In [ ]:
# Asumimos que 'df_final_con_spei' es el DataFrame recién calculado 

print("--- INICIANDO VERIFICACIÓN DE LOS RESULTADOS DEL SPEI ---")

variables_spei = ['spei_3', 'spei_6', 'spei_12']

# --- Test 1: Estadísticas Descriptivas Generales ---
# Un SPEI bien calculado debe tener una media cercana a 0 y una desviación estándar cercana a 1.
# También verificamos los valores mínimos y máximos para ver si están en un rango plausible.
print("\n--- TEST 1: Estadísticas Descriptivas del SPEI (Global) ---")
print(df_final_con_spei[variables_spei].describe())

# --- Test 2: Estadísticas por Estación ---
# Verificamos que la media y la desviación estándar también se cumplan aproximadamente
# para cada estación individual.
print("\n--- TEST 2: Media del SPEI por Estación ---")
print(df_final_con_spei.groupby('estacion_id')[variables_spei].mean())

print("\n--- TEST 3: Desviación Estándar del SPEI por Estación ---")
print(df_final_con_spei.groupby('estacion_id')[variables_spei].std())


# --- Test 4: Verificación de la Paradoja del Sur Oriental ---
# Ahora que el SPEI está corregido, el promedio del SPEI-12 en el sur debería ser más alto (más húmedo) que en el resto de la Región Oriental, como sugerían los datos de P y T.

# Clasificar las estaciones de la Región Oriental
estaciones_sur = ['CAAZ', 'CMEZA', 'PILAR']
df_final_con_spei['grupo_oriental'] = np.where(
    (df_final_con_spei['region'] == 'Oriental') & (df_final_con_spei['estacion_id'].isin(estaciones_sur)),
    'Sur Oriental',
    np.where(df_final_con_spei['region'] == 'Oriental', 'Resto Oriental', 'Occidental')
)

# Calcular el promedio de SPEI-12 por grupo
promedio_spei_grupo = df_final_con_spei.groupby('grupo_oriental')['spei_12'].mean()
print("\n--- TEST 4: Verificación de la Paradoja del Sur (Promedio SPEI-12 por Grupo) ---")
print(promedio_spei_grupo)

--- INICIANDO VERIFICACIÓN DE LOS RESULTADOS DEL SPEI ---

--- TEST 1: Estadísticas Descriptivas del SPEI (Global) ---
            spei_3       spei_6      spei_12
count  3146.000000  3113.000000  3047.000000
mean          -inf     0.000009         -inf
std            NaN     1.000162          NaN
min           -inf    -3.721724         -inf
25%      -0.680620    -0.688097    -0.681917
50%      -0.013244     0.010210     0.004300
75%       0.667298     0.671914     0.659819
max       3.539787     3.282563     3.038758

--- TEST 2: Media del SPEI por Estación ---
                   spei_3    spei_6       spei_12
estacion_id                                      
CAAZ         3.328486e-05  0.000063 -2.049095e-07
CMEZA       -1.208850e-05  0.000003  3.494354e-05
CONC        -2.041366e-04 -0.000009  1.461658e-05
COVIE        6.499047e-05 -0.000001          -inf
GBRU                 -inf -0.000004 -5.271095e-06
LUQUE       -4.677153e-05 -0.000004  3.931417e-07
MCAL        -1.031703e-04 -0.00

/home/hjvargaso/proyectos/tesis_sequia/.venv/lib/python3.12/site-packages/pandas/core/nanops.py:1016: RuntimeWarning: invalid value encountered in subtract
  sqr = _ensure_numeric((avg - values) ** 2)
/home/hjvargaso/proyectos/tesis_sequia/.venv/lib/python3.12/site-packages/pandas/core/nanops.py:1016: RuntimeWarning: invalid value encountered in subtract
  sqr = _ensure_numeric((avg - values) ** 2)


Diagnóstico del Problema: Valores Infinitos (-inf)
Mira los resultados de los tests:

Test 1: mean y min para spei_3 y spei_12 son -inf (infinito negativo). std es NaN.

Test 2: GBRU tiene -inf para spei_3, COVIE para spei_12.

Test 4: Resto Oriental tiene -inf para el promedio de spei_12.

¿Qué está pasando?

La función norm.ppf(cdf) es la que transforma la probabilidad acumulada (CDF, un valor entre 0 y 1) en un Z-score (el SPEI).
Si el valor de la CDF es exactamente 0.0, entonces norm.ppf(0) devuelve -inf (infinito negativo).
Si el valor de la CDF es exactamente 1.0, entonces norm.ppf(1) devuelve +inf (infinito positivo).
Esto ocurre cuando un valor en tu serie balance_acumulado es tan extremo (tan bajo o tan alto) que cae completamente fuera del rango de la distribución pearson3 que fue ajustada. La librería le asigna una probabilidad de 0 o 1.
Esto es un problema numérico y estadístico. No podemos tener infinitos en nuestros datos.

La Solución: "Recortar" la CDF (Climatological Probability Plotting Position)

La solución estándar en la literatura climatológica para evitar este problema es aplicar una pequeña corrección a la CDF antes de la transformación final. Nunca se usan los valores puros de 0 o 1, sino que se "recortan" a un valor muy cercano.
Una de las fórmulas más comunes es la de Gringorten (1963). Se ajusta la probabilidad P (nuestro cdf) de la siguiente manera:
P_ajustada = (P - 0.44) / (n + 0.12) ... No, espera, esta fórmula es para el cálculo de la CDF empírica, no para corregir la teórica.
La solución es más simple: vamos a "recortar" la CDF para que nunca sea exactamente 0 o 1.

**Código Corregido con Recorte de CDF**

In [ ]:
from scipy.stats import pearson3, norm # Importamos las distribuciones correctas

# BUCLE PRINCIPAL PARA EL CÁLCULO DEL SPEI (CON PEARSON III) 

print("--- Iniciando el cálculo del SPEI con la distribución Pearson III ---")

df_completo = pd.merge(df_mensual, df_metadata, on='estacion_id')
df_agrupado = df_completo.set_index('fecha').groupby('estacion_id')

lista_spei_final = []

for estacion_nombre, datos_estacion in df_agrupado:
    
    latitud_estacion = datos_estacion['latitud'].iloc[0]
    temp_series = datos_estacion['temp_media'].fillna(datos_estacion['temp_media'].mean())
    precip_series = datos_estacion['lluvia_total'].fillna(0)
    
    pet_series = thornthwaite_calculo(t_avg=temp_series, lat=latitud_estacion)
    
    datos_estacion['pet'] = pet_series
    datos_estacion['balance_hidrico'] = precip_series - pet_series
    balance = datos_estacion['balance_hidrico'].fillna(0)
    
    # Bucle para cada escala temporal
    for escala in [3, 6, 12]:
        # Acumular el balance hídrico
        balance_acumulado = balance.rolling(window=escala, min_periods=escala).sum()
        series_a_ajustar = balance_acumulado.dropna()
        
        # AJUSTE CON LA DISTRIBUCIÓN CORRECTA 
        # 1. Ajustar la distribución de Pearson Tipo III, esta distribución se define por 3 parámetros: skew (asimetría), loc (media), scale (desv. est.)
        params = pearson3.fit(series_a_ajustar)
        
        # 2. Calcular la CDF (Cumulative Distribution Function)
        cdf = pearson3.cdf(balance_acumulado, *params)
        
        # 3. Transformar la CDF a Z-scores (SPEI)
        # CORRECCIÓN CLAVE PARA EVITAR INFINITOS 
        # "Recortamos" los valores de la CDF para que estén dentro de un rango muy cercano a (0, 1) pero sin tocarlos.
        # 0.0000001 es un valor pequeño y seguro.
        cdf = np.clip(cdf, 0.000001, 0.999999)
        
        # Transformar la CDF a Z-scores (SPEI)
        spei_values = norm.ppf(cdf)
        
        datos_estacion[f'spei_{escala}'] = spei_values
        
    lista_spei_final.append(datos_estacion)
    print(f"SPEI calculado para la estación: {estacion_nombre}")

# Unir todos los resultados
df_final_con_spei = pd.concat(lista_spei_final).reset_index()

print("\n--- Cálculo del SPEI finalizado ---")

# Verificar las últimas filas del nuevo DataFrame
print("\nÚltimas filas del DataFrame con el SPEI correctamente calculado:")
print(df_final_con_spei[['estacion_id', 'fecha', 'lluvia_total', 'temp_media', 'pet', 'spei_3', 'spei_6', 'spei_12']].tail())


--- Iniciando el cálculo del SPEI con la distribución Pearson III ---
SPEI calculado para la estación: CAAZ
SPEI calculado para la estación: CMEZA
SPEI calculado para la estación: CONC
SPEI calculado para la estación: COVIE
SPEI calculado para la estación: GBRU
SPEI calculado para la estación: LUQUE
SPEI calculado para la estación: MCAL
SPEI calculado para la estación: MINGA
SPEI calculado para la estación: PILAR
SPEI calculado para la estación: PJC
SPEI calculado para la estación: SEST

--- Cálculo del SPEI finalizado ---

Últimas filas del DataFrame con el SPEI correctamente calculado:
     estacion_id      fecha  lluvia_total  temp_media         pet    spei_3  \
3163        SEST 2023-08-31          96.9   22.619355  102.317000 -0.706142   
3164        SEST 2023-09-30         141.3   24.970000  120.157514 -0.454075   
3165        SEST 2023-10-31         171.5   25.880645  129.777737 -0.103260   
3166        SEST 2023-11-30         404.6   26.603333  126.336614  1.245491   
3167      

**Verificar de nuevo**

In [ ]:
# Asumimos que 'df_final_con_spei' es el DataFrame recién calculado 

print("--- INICIANDO VERIFICACIÓN DE LOS RESULTADOS DEL SPEI ---")

variables_spei = ['spei_3', 'spei_6', 'spei_12']

# Test 1: Estadísticas Descriptivas Generales 
# Un SPEI bien calculado debe tener una media cercana a 0 y una desviación estándar cercana a 1.
# También verificamos los valores mínimos y máximos para ver si están en un rango plausible.
print("\n--- TEST 1: Estadísticas Descriptivas del SPEI (Global) ---")
print(df_final_con_spei[variables_spei].describe())

# Test 2: Estadísticas por Estación 
# Verificamos que la media y la desviación estándar también se cumplan aproximadamente para cada estación individual.
print("\n--- TEST 2: Media del SPEI por Estación ---")
print(df_final_con_spei.groupby('estacion_id')[variables_spei].mean())

print("\n--- TEST 3: Desviación Estándar del SPEI por Estación ---")
print(df_final_con_spei.groupby('estacion_id')[variables_spei].std())


# Test 4: Verificación de la Paradoja del Sur Oriental 
# Ahora que el SPEI está corregido, el promedio del SPEI-12 en el sur debería ser más alto (más húmedo) que en el resto de la Región Oriental, como sugerían los datos de P y T.

# Clasificar las estaciones de la Región Oriental
estaciones_sur = ['CAAZ', 'CMEZA', 'PILAR']
df_final_con_spei['grupo_oriental'] = np.where(
    (df_final_con_spei['region'] == 'Oriental') & (df_final_con_spei['estacion_id'].isin(estaciones_sur)),
    'Sur Oriental',
    np.where(df_final_con_spei['region'] == 'Oriental', 'Resto Oriental', 'Occidental')
)

# Calcular el promedio de SPEI-12 por grupo
promedio_spei_grupo = df_final_con_spei.groupby('grupo_oriental')['spei_12'].mean()
print("\n--- TEST 4: Verificación de la Paradoja del Sur (Promedio SPEI-12 por Grupo) ---")
print(promedio_spei_grupo)

--- INICIANDO VERIFICACIÓN DE LOS RESULTADOS DEL SPEI ---

--- TEST 1: Estadísticas Descriptivas del SPEI (Global) ---
            spei_3       spei_6      spei_12
count  3146.000000  3113.000000  3047.000000
mean     -0.001464     0.000009    -0.004318
std       1.003750     1.000162     1.011274
min      -4.753424    -3.721724    -4.753424
25%      -0.680620    -0.688097    -0.681917
50%      -0.013244     0.010210     0.004300
75%       0.667298     0.671914     0.659819
max       3.539787     3.282563     3.038758

--- TEST 2: Media del SPEI por Estación ---
                   spei_3    spei_6       spei_12
estacion_id                                      
CAAZ         3.328486e-05  0.000063 -2.049095e-07
CMEZA       -1.208850e-05  0.000003  3.494354e-05
CONC        -2.041366e-04 -0.000009  1.461658e-05
COVIE        6.499047e-05 -0.000001 -4.749505e-02
GBRU        -1.576827e-02 -0.000004 -5.271095e-06
LUQUE       -4.677153e-05 -0.000004  3.931417e-07
MCAL        -1.031703e-04 -0.00

### Guardar resultados

In [12]:
# 'df_final_con_spei' es el DataFrame con los resultados

# Definir el nombre del archivo de salida
nombre_archivo_final = 'datos_mensuales_con_spei.csv'

# Seleccionar columnas importantes para guardar
# Mantenemos las columnas originales y las nuevas del SPEI
columnas_a_guardar = [
    'estacion_id', 
    'fecha', 
    'lluvia_total', 
    'temp_media', 
    'temp_max', 
    'temp_min',
    'pet', # Guardamos también la ETP calculada
    'spei_3', 
    'spei_6', 
    'spei_12'
]

if 'df_final_con_spei' in globals():
    # Crear una copia con solo las columnas que queremos guardar
    df_para_guardar = df_final_con_spei[columnas_a_guardar].copy()

    # Guardar el DataFrame en un archivo CSV
    # index=False evita que pandas guarde el índice del DataFrame como una columna en el archivo
    df_para_guardar.to_csv(nombre_archivo_final, index=False, float_format='%.4f')

    print(f"DataFrame final con SPEI guardado exitosamente en: '{nombre_archivo_final}'")
    print(f"Dimensiones del archivo guardado: {df_para_guardar.shape}")
    print(f"Columnas guardadas: {df_para_guardar.columns.tolist()}")
else:
    print("Error: 'df_final_con_spei' no está definido. Por favor, ejecuta primero la celda de cálculo del SPEI.")

DataFrame final con SPEI guardado exitosamente en: 'datos_mensuales_con_spei.csv'
Dimensiones del archivo guardado: (3168, 10)
Columnas guardadas: ['estacion_id', 'fecha', 'lluvia_total', 'temp_media', 'temp_max', 'temp_min', 'pet', 'spei_3', 'spei_6', 'spei_12']


In [ ]:
import pandas as pd
from tabulate import tabulate

# 1. Cargar los datos desde el archivo CSV 
try:
    df_final_con_spei = pd.read_csv('datos_mensuales_con_spei.csv')
    print("Archivo 'datos_mensuales_con_spei.csv' cargado exitosamente.")
except FileNotFoundError:
    print("Error: El archivo 'datos_mensuales_con_spei.csv' no se encontró.")
    print("Por favor, asegúrate de que el archivo esté en el mismo directorio que este script.")
    exit() # Detiene el script si no se encuentra el archivo


# 2. Preparar el DataFrame para la tabla 
# Seleccionar y renombrar columnas para que sean más cortas
columnas_tabla = {
    'estacion_id': 'Est.',
    'fecha': 'Fecha',
    'lluvia_total': 'P (mm)',
    'temp_media': 'T (°C)',
    'pet': 'ETP (mm)',
    'spei_3': 'SPEI-3',
    'spei_6': 'SPEI-6',
    'spei_12': 'SPEI-12'
}
df_tabla_spei = df_final_con_spei[columnas_tabla.keys()].copy()
df_tabla_spei.rename(columns=columnas_tabla, inplace=True)

# Formatear la fecha a un formato más corto (Año-Mes)
df_tabla_spei['Fecha'] = pd.to_datetime(df_tabla_spei['Fecha']).dt.strftime('%Y-%m')

# Formatear los números para que ocupen menos espacio
df_tabla_spei['P (mm)'] = df_tabla_spei['P (mm)'].map('{:.1f}'.format)
df_tabla_spei['T (°C)'] = df_tabla_spei['T (°C)'].map('{:.1f}'.format)
df_tabla_spei['ETP (mm)'] = df_tabla_spei['ETP (mm)'].map('{:.1f}'.format)
df_tabla_spei['SPEI-3'] = df_tabla_spei['SPEI-3'].map('{:.2f}'.format)
df_tabla_spei['SPEI-6'] = df_tabla_spei['SPEI-6'].map('{:.2f}'.format)
df_tabla_spei['SPEI-12'] = df_tabla_spei['SPEI-12'].map('{:.2f}'.format)

# Reemplazar los 'nan' de los SPEI iniciales por un guion para mayor claridad
df_tabla_spei.replace('nan', '-', inplace=True)

#  3. Generar el Código LaTeX con 'longtable'
# Primero, generamos el interior de la tabla
interior_tabla = tabulate(
    df_tabla_spei,
    headers="keys",
    tablefmt="latex_booktabs",
    showindex=False
)

# Ahora, envolvemos el 'tabular' generado en un entorno 'longtable' y añadimos un encabezado que se repita en cada página

lineas = interior_tabla.split('\n')
cuerpo = "\n".join(lineas[4:-1]) # Las filas de datos

codigo_longtable = f"""
\\begin{{longtable}}{{l l r r r r r r}}
\\caption{{Datos mensuales finales y valores de SPEI calculados para las 11 estaciones seleccionadas. P: Precipitación; T: Temperatura Media; ETP: Evapotranspiración Potencial.}} \\\\
\\label{{tab:datos_spei_completos}} \\\\

% --- Encabezado que se repite en cada página ---
\\toprule
\\textbf{{Est.}} & \\textbf{{Fecha}} & \\textbf{{P (mm)}} & \\textbf{{T (°C)}} & \\textbf{{ETP (mm)}} & \\textbf{{SPEI-3}} & \\textbf{{SPEI-6}} & \\textbf{{SPEI-12}} \\\\
\\midrule
\\endfirsthead

\\multicolumn{{8}}{{c}}{{\\tablename\\ \\thetable{{}} -- continuación de la página anterior}} \\\\
\\toprule
\\textbf{{Est.}} & \\textbf{{Fecha}} & \\textbf{{P (mm)}} & \\textbf{{T (°C)}} & \\textbf{{ETP (mm)}} & \\textbf{{SPEI-3}} & \\textbf{{SPEI-6}} & \\textbf{{SPEI-12}} \\\\
\\midrule
\\endhead

% --- Aquí empieza el cuerpo de la tabla ---
{cuerpo}
% ---------------------------------------

\\bottomrule
\\end{{longtable}}
"""

# Guardar en un archivo para copiar y pegar fácilmente
nombre_archivo_salida = "tabla_spei_apendice.tex"
with open(nombre_archivo_salida, "w") as f:
    f.write(codigo_longtable)

print(f"\n¡Hecho! El código LaTeX para la tabla larga ha sido guardado en '{nombre_archivo_salida}'")

Archivo 'datos_mensuales_con_spei.csv' cargado exitosamente.

¡Hecho! El código LaTeX para la tabla larga ha sido guardado en 'tabla_spei_apendice.tex'
